In [13]:
!pip install -U langchain_community langchainhub chromadb langchain langchain-google-genai langchain-text-splitters langsmith

In [39]:
from google.colab import userdata
import os

# langchain-google-genai looks for GOOGLE_API_KEY specifically
os.environ["GOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [15]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(web_paths=["https://docs.langchain.com/oss/python/langgraph/overview"])
docs = loader.load()

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
print(len(splits))

11


In [27]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=userdata.get("GEMINI_API_KEY")
)
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)

print(vectorstore._collection.count())

11


In [28]:
retriever = vectorstore.as_retriever()

In [29]:
!pip install langsmith

from langsmith import Client

ls_client = Client()
prompt = ls_client.pull_prompt("rlm/rag-prompt", dangerously_pull_public_prompt=True)

In [33]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=userdata.get("GEMINI_API_KEY")
)

In [35]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [38]:
rag_chain.invoke("What is langgraph?")

'LangGraph is a low-level orchestration framework and runtime for building, managing, and deploying long-running, stateful agents. It gives developers fine-grained control to combine deterministic, hand-coded steps with LLM-driven agentic steps in the same graph. Additionally, it focuses on essential agent capabilities such as durable execution, streaming, human-in-the-loop interactions, and persistence.'